In [23]:
import pandas as pd
import numpy as np
from sklearn.utils import Bunch

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

In [18]:
data = pd.read_csv("clean_data.csv")
data['Is_Fraud'] = data['Is_Fraud'].map({0: 'No Fraud', 1: 'Fraud'})
data.head()

,Unnamed: 0,Transaction_ID,Transaction_Amount (in Million),Transaction_Time,Transaction_Date,Transaction_Type,Merchant_Category,Transaction_Location,Customer_Home_Location,Distance_From_Home,...,Daily_Transaction_Count,Weekly_Transaction_Count,Avg_Transaction_Amount (in Million),Max_Transaction_Last_24h (in Million),Is_International_Transaction,Is_New_Merchant,Failed_Transaction_Count,Unusual_Time_Transaction,Previous_Fraud_Count,Is_Fraud
0,0,431438.0,6.0,10:54,2025-03-08,POS,ATM,Singapore,Lahore,466.0,...,4.0,17.0,2.0,4.0,1,1,0.0,0,1.0,No Fraud
1,1,902451.0,9.0,19:23,2025-01-17,ATM,ATM,Singapore,Lahore,215.0,...,4.0,9.0,5.0,8.0,1,1,1.0,0,1.0,No Fraud
2,2,223410.0,3.0,10:20,2025-04-30,POS,Electronics,Faisalabad,Faisalabad,216.0,...,5.0,18.0,5.0,8.0,1,0,0.0,1,1.0,No Fraud
3,3,145626.0,1.0,14:11,2025-02-21,Online,Grocery,London,Karachi,408.0,...,6.0,18.0,5.0,1.0,0,1,2.0,1,1.0,No Fraud
4,4,414637.0,1.0,04:12,2025-04-11,Online,Electronics,Singapore,Islamabad,209.0,...,3.0,18.0,4.0,3.0,0,1,1.0,0,1.0,No Fraud


In [26]:
features = data.drop('Is_Fraud', axis=1)
features = data.drop('Transaction_Time', axis=1)
target = data['Is_Fraud']

# Package into a Bunch
fraud_data = Bunch(
    data=features.values,
    target=target.values,
    feature_names=features.columns.tolist(),
    target_names=["No Fraud", "Fraud"],
)

X = pd.DataFrame(fraud_data.data, columns=fraud_data.feature_names)
y = pd.Series(fraud_data.target, name="target")

target_names = fraud_data.target_names

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("\nTarget names:", target_names)
print("\nClass counts:")
print(y.value_counts().sort_index())

Feature matrix shape: (49876, 22)
Target vector shape: (49876,)

Target names: ['No Fraud', 'Fraud']

Class counts:
target
Fraud        2418
No Fraud    47458
Name: count, dtype: int64


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Training set shape: (37407, 22)
Test set shape: (12469, 22)


In [28]:
def evaluate_classifier(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    results = {
        "model": model_name,
        "train_accuracy": train_acc,
        "test_accuracy": test_acc
    }

    if hasattr(model, "predict_proba"):
        test_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, test_proba)
        roc_auc = auc(fpr, tpr)
        results["roc_auc"] = roc_auc
    else:
        results["roc_auc"] = np.nan

    return model, results

In [29]:
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree, tree_results = evaluate_classifier(tree, X_train, X_test, y_train, y_test, "Decision Tree")
pd.DataFrame([tree_results])

ValueError: could not convert string to float: '2025-01-10'